## Graph data

In [3]:
!pip install pandas plotly numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Carica i file CSV
perf_df = pd.read_csv('stopwatch_runs.csv')
fake_df = pd.read_csv('fake_dataset.csv')

# Analizza i tipi di colonna dal dataset
print("Tipi di colonne in fake_dataset.csv:")
print(fake_df.dtypes)
print("\nValori unici per f4:", fake_df['f4'].nunique())
print("Valori unici per f5:", fake_df['f5'].nunique())
print("\nDistribuzione f4:", fake_df['f4'].value_counts().index.tolist())
print("Distribuzione f5:", fake_df['f5'].value_counts().index.tolist())

Tipi di colonne in fake_dataset.csv:
f1        float64
f2          int64
f3         object
f4         object
f5          int64
date       object
target      int64
dtype: object

Valori unici per f4: 3
Valori unici per f5: 6

Distribuzione f4: ['cherry', 'banana', 'apple']
Distribuzione f5: [1, 2, 3, 0, 5, 4]


In [5]:
# Estrai affected_features da strategy_json
perf_df['affected_feature'] = perf_df['strategy_json'].apply(
    lambda x: json.loads(x.replace('""', '"'))['affected_features'][0]
)

# Definisci i tipi di colonna per la legenda
column_types = {
    'f1': 'float64',
    'f2': 'int64',
    'f3': 'categorical (3 classi)',
    'f4': 'categorical (3 classi)',
    'f5': 'int64 (6 classi)',
    'date': 'datetime',
    'target': 'binary'
}

print("Dati estratti:")
print(perf_df[['metodo', 'backend', 'affected_feature', 'stopwatch_sec']].head(20))

Dati estratti:
        metodo backend affected_feature  stopwatch_sec
0   duplicated  PANDAS               f1       0.512010
1   duplicated   SPARK               f1       6.404904
2   duplicated  PANDAS               f2       0.497250
3   duplicated   SPARK               f2       1.795457
4   duplicated  PANDAS               f3       0.500013
5   duplicated   SPARK               f3       1.651567
6   duplicated  PANDAS               f4       0.502125
7   duplicated   SPARK               f4       1.483769
8   duplicated  PANDAS               f5       0.493062
9   duplicated   SPARK               f5       1.464234
10  duplicated  PANDAS             date       0.493285
11  duplicated   SPARK             date       1.437912
12  duplicated  PANDAS           target       0.500428
13  duplicated   SPARK           target       1.438766
14     missing  PANDAS               f1       0.177129
15     missing   SPARK               f1       2.298614
16     missing  PANDAS               f2       0.17

In [6]:
# Crea due grafici separati per backend (SPARK e PANDAS)

# Mappa colori per tipo di colonna
color_map = {
    'f1': '#d62728',      # rosso (float)
    'f2': '#f7b801',      # giallo (int)
    'f3': '#2ca02c',      # verde (categorical)
    'f4': '#1f77b4',      # blu (categorical)
    'f5': '#17becf',      # ciano (int)
    'date': '#9467bd',    # viola (datetime)
    'target': '#8c564b'   # marrone (binary)
}

# Crea figura con 2 sottografici
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Backend: SPARK", "Backend: PANDAS"),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

metodi = ['outlier', 'duplicated', 'missing', 'noise']
affected_features = ['f1', 'f2', 'f3', 'f4', 'f5', 'date', 'target']
backends = ['SPARK', 'PANDAS']

# Per ogni backend
for row_idx, backend in enumerate(backends, start=1):
    backend_data = perf_df[perf_df['backend'] == backend]
    
    # Aggiungi barre per ogni colonna
    for feature in affected_features:
        feature_data = backend_data[backend_data['affected_feature'] == feature]
        
        # Calcola i valori medi per ogni metodo
        metodo_values = []
        for metodo in metodi:
            metodo_data = feature_data[feature_data['metodo'] == metodo]['stopwatch_sec'].values
            metodo_values.append(np.mean(metodo_data) if len(metodo_data) > 0 else 0)
        
        fig.add_trace(
            go.Bar(
                x=metodi,
                y=metodo_values,
                name=f'{feature} ({column_types[feature]})',
                marker=dict(color=color_map[feature]),
                text=[f'{v:.2f}' for v in metodo_values],
                textposition='auto',
                legendgroup=feature,
                showlegend=(row_idx == 1),  # Mostra legenda solo per il primo grafico
            ),
            row=row_idx, col=1
        )

# Calcola il massimo valore da tutti i dati per fissare la scala
max_value = perf_df['stopwatch_sec'].max()
y_max = max_value * 1.1  # Aggiunge il 10% di spazio sopra

fig.update_xaxes(title_text="Metodo", row=2, col=1)
fig.update_xaxes(title_text="", row=1, col=1)

# Applica la stessa scala Y per entrambi i grafici
fig.update_yaxes(title_text="Tempo (sec)", range=[0, y_max], row=1, col=1)
fig.update_yaxes(title_text="Tempo (sec)", range=[0, y_max], row=2, col=1)

fig.update_layout(
    title='Confronto Performance: SPARK vs PANDAS (1.000.000 records)',
    height=900,
    width=1200,
    font=dict(size=11),
    hovermode='x unified',
    barmode='group'
)

fig.show()

In [7]:
# Carica il file stress_test.csv
stress_df = pd.read_csv('/home/cava/Documents/Repos/python/pucktrick/performance_test/stress_test.csv')

# Estrai affected_features da strategy_json
stress_df['affected_feature'] = stress_df['strategy_json'].apply(
    lambda x: json.loads(x.replace('""', '"'))['affected_features'][0]
)

print("Dati stress test:")
print(stress_df[['metodo', 'backend', 'iterazione', 'num_righe', 'stopwatch_sec_iter', 'stopwatch_sec_tot']].head(20))
print("\nMetodi unici:", stress_df['metodo'].unique())
print("Backend unici:", stress_df['backend'].unique())

Dati stress test:
     metodo backend  iterazione  num_righe  stopwatch_sec_iter  \
0  outliers   SPARK           1    1000000            3.584395   
1  outliers   SPARK           2    2000000            3.042496   
2  outliers   SPARK           3    4000000            3.510963   
3  outliers   SPARK           4    8000000            5.853660   
4  outliers   SPARK           5   16000000           11.021906   
5  outliers   SPARK           6   32000000           20.675027   
6  outliers   SPARK           7   64000000           41.238236   
7  outliers   SPARK           8  128000000           84.641408   

   stopwatch_sec_tot  
0           3.584395  
1           6.626892  
2          10.137855  
3          15.991515  
4          27.013421  
5          47.688449  
6          88.926685  
7         173.568093  

Metodi unici: ['outliers']
Backend unici: ['SPARK']


In [8]:
# Grafico stress test: Tempo vs Numero di righe per outliers

# Raggruppa per backend
backends_stress = stress_df['backend'].unique()

if len(backends_stress) == 1:
    # Se c'è un solo backend, crea un unico grafico
    backend = backends_stress[0]
    backend_data = stress_df[stress_df['backend'] == backend].sort_values('num_righe')
    
    fig_stress = go.Figure()
    
    fig_stress.add_trace(go.Scatter(
        x=backend_data['num_righe'],
        y=backend_data['stopwatch_sec_iter'],
        mode='lines+markers',
        name='Tempo per iterazione',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=8)
    ))
    
    fig_stress.update_layout(
        title=f'Stress Test: Tempo di calcolo vs Numero di righe ({backend})',
        xaxis_title='Numero di righe',
        yaxis_title='Tempo (secondi)',
        height=600,
        width=1000,
        font=dict(size=12),
        hovermode='x unified'
    )
    
else:
    # Se ci sono più backend, crea 2 grafici
    fig_stress = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Backend: {b}' for b in sorted(backends_stress)],
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    for col_idx, backend in enumerate(sorted(backends_stress), start=1):
        backend_data = stress_df[stress_df['backend'] == backend].sort_values('num_righe')
        
        fig_stress.add_trace(
            go.Scatter(
                x=backend_data['num_righe'],
                y=backend_data['stopwatch_sec_iter'],
                mode='lines+markers',
                name=backend,
                line=dict(width=3),
                marker=dict(size=8),
                showlegend=(col_idx == 1)
            ),
            row=1, col=col_idx
        )
        
        fig_stress.update_xaxes(title_text="Numero di righe", row=1, col=col_idx)
        fig_stress.update_yaxes(title_text="Tempo (sec)", row=1, col=col_idx)
    
    fig_stress.update_layout(
        title='Stress Test: Tempo di calcolo vs Numero di righe (outliers)',
        height=500,
        width=1400,
        font=dict(size=11),
        hovermode='x unified'
    )

fig_stress.show()

## Break-even analysis — `breakeven_runs.csv`

**Plot 1**: per ogni metodo di noise injection, tempo di esecuzione Pandas vs Spark al crescere delle righe. La linea rossa tratteggiata indica il **break-even point** (prima iterazione in cui Spark supera Pandas).  
**Plot 2**: per ogni iterazione, istogramma raggruppato e stacked che mostra la **somma dei tempi** di tutti i metodi per backend (PANDAS affiancato a SPARK), con il contributo di ogni metodo annotato dentro la barra.


In [9]:
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

be_df = pd.read_csv('breakeven_runs.csv')
be_df = be_df[be_df['stopwatch_sec'] >= 0].copy()

METHODS  = ['duplicated', 'missing', 'noise', 'outlier', 'labels']
BACKENDS = ['PANDAS', 'SPARK']

# ── Calcola il punto ESATTO di incrocio per interpolazione lineare in log-space
def find_crossing(pd_series, sp_series):
    """
    Restituisce (x_cross, y_cross) interpolati tra i due campioni consecutivi
    in cui la differenza (Pandas - Spark) cambia segno.
    Restituisce None se non esiste incrocio nei dati.
    """
    rows = sorted(pd_series.index)
    for i in range(len(rows) - 1):
        r0, r1 = rows[i], rows[i+1]
        d0 = pd_series[r0] - sp_series[r0]
        d1 = pd_series[r1] - sp_series[r1]
        if d0 * d1 < 0:   # cambio di segno → incrocio in questo intervallo
            frac   = d0 / (d0 - d1)
            log_cx = np.log10(r0) + frac * (np.log10(r1) - np.log10(r0))
            cx     = 10 ** log_cx
            cy     = pd_series[r0] + frac * (pd_series[r1] - pd_series[r0])
            return cx, cy
    return None

crossing_points = {}   # metodo → (x_rows, y_sec) | None
for metodo in METHODS:
    d  = be_df[be_df['metodo'] == metodo].sort_values('num_righe')
    ps = d[d['backend']=='PANDAS'].set_index('num_righe')['stopwatch_sec']
    ss = d[d['backend']=='SPARK' ].set_index('num_righe')['stopwatch_sec']
    crossing_points[metodo] = find_crossing(ps, ss)

print('Crossing points:')
for m, cp in crossing_points.items():
    if cp:
        print(f'  {m}: {cp[0]/1e6:.2f}M rows  @ {cp[1]:.2f}s')
    else:
        # Se Spark è già più veloce al primo punto → BE avviene prima della prima misura
        d  = be_df[be_df['metodo']==m].sort_values('num_righe')
        ps = d[d['backend']=='PANDAS'].set_index('num_righe')['stopwatch_sec']
        ss = d[d['backend']=='SPARK' ].set_index('num_righe')['stopwatch_sec']
        first_row = sorted(ps.index)[0]
        if ss[first_row] < ps[first_row]:
            print(f'  {m}: Spark già più veloce al primo punto ({first_row/1e6:.0f}M rows) — BE < prima misura')
        else:
            print(f'  {m}: nessun incrocio rilevato (Pandas sempre più veloce)')


Crossing points:
  duplicated: nessun incrocio rilevato (Pandas sempre più veloce)
  missing: 5.43M rows  @ 0.75s
  noise: Spark già più veloce al primo punto (1M rows) — BE < prima misura
  outlier: Spark già più veloce al primo punto (1M rows) — BE < prima misura
  labels: Spark già più veloce al primo punto (1M rows) — BE < prima misura


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT 1 — Break-even point per metodo
# ══════════════════════════════════════════════════════════════════════════════

COLOR_PANDAS = '#1f77b4'
COLOR_SPARK  = '#ff7f0e'
COLOR_BE     = '#d62728'

fig1 = make_subplots(
    rows=2, cols=5,
    # Riga 1: grafici | Riga 2: caption break-even (altezza minima)
    row_heights=[0.88, 0.12],
    subplot_titles=[m.capitalize() for m in METHODS] + ['']*5,
    shared_yaxes=False,
    horizontal_spacing=0.06,
    vertical_spacing=0.04,
)

for col_idx, metodo in enumerate(METHODS, start=1):
    d  = be_df[be_df['metodo'] == metodo].sort_values('num_righe')
    ps = d[d['backend']=='PANDAS'].set_index('num_righe')['stopwatch_sec']
    ss = d[d['backend']=='SPARK' ].set_index('num_righe')['stopwatch_sec']
    first_row = sorted(ps.index)[0]
    spark_faster_from_start = (ss[first_row] < ps[first_row])
    cp = crossing_points[metodo]

    # ── Tracce Pandas e Spark ────────────────────────────────────────────────
    for backend, series, color, dash in [
        ('PANDAS', ps, COLOR_PANDAS, 'solid'),
        ('SPARK',  ss, COLOR_SPARK,  'dot'),
    ]:
        fig1.add_trace(
            go.Scatter(
                x=list(series.index),
                y=list(series.values),
                mode='lines+markers',
                name=backend,
                legendgroup=backend,
                showlegend=(col_idx == 1),
                line=dict(color=color, width=2.5, dash=dash),
                marker=dict(size=7),
                hovertemplate=(
                    f'<b>{backend}</b><br>'
                    'Righe: %{x:,}<br>Tempo: %{y:.3f}s<extra></extra>'
                ),
            ),
            row=1, col=col_idx,
        )

    # ── Marker e linea break-even ─────────────────────────────────────────────
    if cp is not None:
        # Incrocio calcolato per interpolazione
        cx, cy = cp
        fig1.add_vline(
            x=cx, line=dict(color=COLOR_BE, width=1.5, dash='dashdot'),
            row=1, col=col_idx,
        )
        fig1.add_trace(
            go.Scatter(
                x=[cx], y=[cy],
                mode='markers',
                marker=dict(color=COLOR_BE, size=11, symbol='x-thin',
                            line=dict(color=COLOR_BE, width=3)),
                showlegend=(col_idx == 1),
                name='Break-even',
                legendgroup='be',
                hovertemplate=(
                    f'<b>Break-even</b><br>'
                    f'{cx/1e6:.2f}M rows — {cy:.2f}s<extra></extra>'
                ),
            ),
            row=1, col=col_idx,
        )
        caption = f'BE ≈ {cx/1e6:.2f}M rows'

    elif spark_faster_from_start:
        # BE avvenuto prima della prima misura → linea sul primo punto
        fig1.add_vline(
            x=first_row, line=dict(color=COLOR_BE, width=1.5, dash='dashdot'),
            row=1, col=col_idx,
        )
        fig1.add_trace(
            go.Scatter(
                x=[first_row],
                y=[(float(ps[first_row]) + float(ss[first_row])) / 2],
                mode='markers',
                marker=dict(color=COLOR_BE, size=11, symbol='x-thin',
                            line=dict(color=COLOR_BE, width=3)),
                showlegend=False,
                legendgroup='be',
                hovertemplate=f'BE < {first_row/1e6:.0f}M rows<extra></extra>',
            ),
            row=1, col=col_idx,
        )
        caption = f'BE < {first_row/1e6:.0f}M rows'

    else:
        # Nessun break-even nei dati
        caption = 'No BE found'

    # ── Caption in riga 2 (scatter invisibile con solo testo) ────────────────
    fig1.add_trace(
        go.Scatter(
            x=[0.5], y=[0.5],
            mode='text',
            text=[f'<b>{caption}</b>'],
            textfont=dict(size=11, color=COLOR_BE),
            showlegend=False,
            hoverinfo='skip',
        ),
        row=2, col=col_idx,
    )

    fig1.update_xaxes(type='log', tickformat='.2s', title_text= '', row=1, col=col_idx)
    fig1.update_yaxes(title_text='Time (s)' if col_idx == 1 else '',
                      row=1, col=col_idx)

    # Riga 2: nasconde assi e griglia — è solo una caption
    fig1.update_xaxes(visible=False, row=2, col=col_idx)
    fig1.update_yaxes(visible=False, row=2, col=col_idx)

fig1.update_layout(
    title=dict(
        text='<b>Break-even point: Pandas vs Spark per ogni metodo di noise injection</b>',
        font=dict(size=16), x=0.5,
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.06,
        xanchor='center', x=0.5, font=dict(size=12),
    ),
    height=470,
    plot_bgcolor='#fafafa', paper_bgcolor='white',
    margin=dict(t=100, b=20, l=60, r=20),
)

fig1.show()


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT 2 — Istogramma raggruppato + stacked (fix: x-label interleaved)
# Tecnica: le etichette sull'asse x includono il backend,
# così le barre PANDAS e SPARK sono naturalmente affiancate e impilate.
# ══════════════════════════════════════════════════════════════════════════════

METHOD_COLORS = {
    'duplicated': '#4e79a7',
    'missing':    '#f28e2b',
    'noise':      '#e15759',
    'outlier':    '#76b7b2',
    'labels':     '#59a14f',
}

iters_sorted   = sorted(be_df['iterazione'].unique())
rows_per_iter  = (
    be_df[['iterazione','num_righe']].drop_duplicates()
    .set_index('iterazione')['num_righe'].to_dict()
)

def fmt_rows(n):
    return f'{n//1_000_000}M' if n >= 1_000_000 else f'{n//1_000}K'

# Asse X interleaved: ['1M PANDAS', '1M SPARK', '2M PANDAS', '2M SPARK', ...]
# Questo fa sì che barmode='stack' crei due colonne distinte per ogni dimensione.
x_all = []
for it in iters_sorted:
    for be in BACKENDS:
        x_all.append(f'{fmt_rows(rows_per_iter[it])}<br>{be}')

fig2 = go.Figure()

for metodo in METHODS:
    for backend in BACKENDS:
        d = (
            be_df[(be_df['metodo']==metodo) & (be_df['backend']==backend)]
            .sort_values('iterazione')
        )
        # Costruisce valori allineati all'asse X interleaved:
        # per ogni posizione X, il valore è 0 se non appartiene a questo backend
        vals  = []
        texts = []
        for it in iters_sorted:
            for be in BACKENDS:
                if be == backend:
                    row = d[d['iterazione']==it]
                    v   = float(row['stopwatch_sec'].iloc[0]) if len(row) > 0 else 0.0
                    vals.append(v)
                    texts.append(f'{v:.1f}s' if v >= 2.0 else '')
                else:
                    vals.append(0.0)
                    texts.append('')

        fig2.add_trace(go.Bar(
            name=f'{metodo} ({backend})',
            x=x_all,
            y=vals,
            text=texts,
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=9, color='white'),
            marker=dict(
                color=METHOD_COLORS[metodo],
                opacity=1.0 if backend=='SPARK' else 0.5,
                pattern=dict(
                    shape='/' if backend=='PANDAS' else '',
                    solidity=0.4,
                ),
                line=dict(color='white', width=0.5),
            ),
            legendgroup=metodo,
            legendgrouptitle_text=metodo.capitalize() if backend=='PANDAS' else None,
            hovertemplate=(
                f'<b>{metodo} — {backend}</b><br>'
                'Righe: %{x}<br>Tempo: %{y:.3f}s<extra></extra>'
            ),
        ))

fig2.update_layout(
    barmode='stack',
    title=dict(
        text=(
            '<b>Tempo totale per iterazione — somma dei metodi per backend</b><br>'
            '<sup>Per ogni dimensione: barra sinistra = PANDAS (tratteggio /), '
            'barra destra = SPARK (pieno) | Segmenti = contributo per metodo</sup>'
        ),
        font=dict(size=15), x=0.5,
    ),
    xaxis=dict(
        title='Dataset size — Backend',
        tickfont=dict(size=10),
        tickangle=0,
    ),
    yaxis=dict(
        title='Tempo totale (s)',
        showgrid=True, gridcolor='#ececec',
    ),
    legend=dict(
        title='Metodo (backend)',
        groupclick='toggleitem',
        font=dict(size=11),
        tracegroupgap=6,
    ),
    height=580,
    plot_bgcolor='#fafafa', paper_bgcolor='white',
    margin=dict(t=110, b=80, l=70, r=20),
    bargap=0.08,
)

fig2.show()


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT 2 — Istogramma raggruppato + stacked con differenza Pandas/Spark
# ══════════════════════════════════════════════════════════════════════════════

METHOD_COLORS = {
    'duplicated': '#4e79a7',
    'missing':    '#f28e2b',
    'noise':      '#e15759',
    'outlier':    '#76b7b2',
    'labels':     '#59a14f',
}

iters_sorted  = sorted(be_df['iterazione'].unique())
rows_per_iter = (
    be_df[['iterazione','num_righe']].drop_duplicates()
    .set_index('iterazione')['num_righe'].to_dict()
)

def fmt_rows(n):
    return f'{n//1_000_000}M' if n >= 1_000_000 else f'{n//1_000}K'

# Asse X interleaved: ['1M PANDAS', '1M SPARK', '2M PANDAS', '2M SPARK', ...]
x_all = []
for it in iters_sorted:
    for be in BACKENDS:
        x_all.append(f'{fmt_rows(rows_per_iter[it])}<br>{be}')

# Pre-calcola totali per iterazione (per le annotazioni di differenza)
iter_totals = {}   # it → {'PANDAS': float, 'SPARK': float}
for it in iters_sorted:
    iter_totals[it] = {}
    for be in BACKENDS:
        iter_totals[it][be] = float(
            be_df[(be_df['iterazione']==it) & (be_df['backend']==be)]['stopwatch_sec'].sum()
        )

# ── Figura con asse Y secondario per il ratio ─────────────────────────────────
fig2 = make_subplots(specs=[[{'secondary_y': True}]])

for metodo in METHODS:
    for backend in BACKENDS:
        d = (
            be_df[(be_df['metodo']==metodo) & (be_df['backend']==backend)]
            .sort_values('iterazione')
        )
        vals, texts = [], []
        for it in iters_sorted:
            for be in BACKENDS:
                if be == backend:
                    row = d[d['iterazione']==it]
                    v   = float(row['stopwatch_sec'].iloc[0]) if len(row) > 0 else 0.0
                    vals.append(v)
                    texts.append(f'{v:.1f}s' if v >= 2.0 else '')
                else:
                    vals.append(0.0)
                    texts.append('')

        fig2.add_trace(go.Bar(
            name=f'{metodo} ({backend})',
            x=x_all,
            y=vals,
            text=texts,
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=9, color='white'),
            marker=dict(
                color=METHOD_COLORS[metodo],
                opacity=1.0 if backend=='SPARK' else 0.5,
                pattern=dict(shape='/' if backend=='PANDAS' else '', solidity=0.4),
                line=dict(color='white', width=0.5),
            ),
            legendgroup=metodo,
            legendgrouptitle_text=metodo.capitalize() if backend=='PANDAS' else None,
            hovertemplate=(
                f'<b>{metodo} — {backend}</b><br>'
                'Righe: %{x}<br>Tempo: %{y:.3f}s<extra></extra>'
            ),
            #legend=False,
        ), secondary_y=False)

# ── Linea ratio (asse Y secondario) ──────────────────────────────────────────
# Posizionata al centro di ogni coppia PANDAS/SPARK sull'asse X interleaved
ratio_x, ratio_y = [], []
for it in iters_sorted:
    tot_pd = iter_totals[it]['PANDAS']
    tot_sp = iter_totals[it]['SPARK']
    # La coppia di barre è a posizioni (2*idx) e (2*idx+1); il centro è fra le due
    idx = iters_sorted.index(it)
    # X come stringa della barra SPARK (seconda della coppia)
    ratio_x.append(f'{fmt_rows(rows_per_iter[it])}<br>SPARK')
    ratio_y.append(round(tot_pd / tot_sp, 1) if tot_sp > 0 else 0)

fig2.add_trace(go.Scatter(
    x=ratio_x,
    y=ratio_y,
    mode='lines+markers+text',
    name='Speedup (Pandas/Spark)',
    line=dict(color='#9467bd', width=2.5, dash='dashdot'),
    marker=dict(size=9, color='#9467bd', symbol='diamond'),
    text=[f'{r:.1f}×' for r in ratio_y],
    textposition='top center',
    textfont=dict(size=11, color='#9467bd'),
    hovertemplate='<b>Speedup</b><br>%{x}<br>Pandas è %{y:.1f}× più lento di Spark<extra></extra>',
    legendgroup='ratio',
    showlegend=True,
), secondary_y=True)

# ── Annotazioni Δ differenza assoluta sopra ogni coppia ──────────────────────
# Appese sopra la barra PANDAS (la più alta) con freccia di bracket
for it in iters_sorted:
    tot_pd = iter_totals[it]['PANDAS']
    tot_sp = iter_totals[it]['SPARK']
    diff   = tot_pd - tot_sp
    # Etichetta compatta: Δ +XXXs
    label  = f'Δ {diff:+.0f}s'
    # X = posizione barra PANDAS (prima della coppia)
    x_pandas_label = f'{fmt_rows(rows_per_iter[it])}<br>PANDAS'
    fig2.add_annotation(
        x=x_pandas_label,
        y=tot_pd,
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowcolor='#555',
        arrowwidth=1.2,
        ax=0, ay=-32,
        font=dict(size=10, color='#333'),
        bgcolor='rgba(255,255,255,0.82)',
        bordercolor='#aaa',
        borderwidth=1,
        borderpad=3,
        yref='y',
    )

fig2.update_layout(
    barmode='stack',
    title=dict(
        text=(
            '<b>Tempo totale per iterazione — somma dei metodi per backend</b><br>'
            '<sup>Barre: PANDAS (tratteggio /) vs SPARK (pieno) | '
            'Linea viola = speedup Pandas/Spark | Etichette Δ = differenza assoluta</sup>'
        ),
        font=dict(size=15), x=0.5,
    ),
    xaxis=dict(title='Dataset size — Backend', tickfont=dict(size=10)),
    yaxis=dict(title='Tempo totale (s)', showgrid=True, gridcolor='#ececec'),
    yaxis2=dict(
        title='Speedup (Pandas / Spark)',
        showgrid=False,
        ticksuffix='×',
        overlaying='y',
        side='right',
        rangemode='tozero',
        color='#9467bd',
    ),
    legend=dict(
        title='Metodo (backend)',
        groupclick='toggleitem',
        font=dict(size=11),
        tracegroupgap=6,
    ),
    height=600,
    plot_bgcolor='#fafafa', paper_bgcolor='white',
    margin=dict(t=110, b=80, l=70, r=80),
    bargap=0.08,
)

fig2.show()
